# 89 — Calibrate SmolVLA source-chunk reconstruction

Model-only diagnostic over 20 fixed notebook-87 roots. It creates no environments and writes no trees. It separates the seven active LIBERO action dimensions from SmolVLA's padded output dimensions, compares reconstruction under GPU batch sizes 1/8/9, sweeps the expected P&P RNG draw offset by ±5, and checks generated-chunk-width versus executed-action-width time conditioning.

In [ ]:
EXTRAS = 'sim'
SETUP_ENV = True
import urllib.request
exec(urllib.request.urlopen('https://raw.githubusercontent.com/ArjunS07/cs159-sp26/main/pnp-vla/scripts/colab_bootstrap.py').read().decode())

In [ ]:
from pnp.smolvla_reconstruction_calibration import run_smolvla_reconstruction_calibration

N_ROOTS = 20
BATCH_SIZES = (1, 8, 9)
DRAW_OFFSETS = tuple(range(-5, 6))
OUTPUT_CSV = '/content/drive/MyDrive/pnp_smolvla_calibration/source_reconstruction_v1.csv'

calibration = run_smolvla_reconstruction_calibration(
    n_roots=N_ROOTS, batch_sizes=BATCH_SIZES, draw_offsets=DRAW_OFFSETS,
    output_csv=OUTPUT_CSV)
calibration.head()

In [ ]:
# Compact tables useful to paste back for interpretation.
display(calibration[calibration.mode.eq('batch_shape')].groupby('batch_size')[
    ['first10_active_rms', 'first10_active_max_abs', 'first10_padded_rms',
     'first10_all_rms']].quantile([0.5, 0.9, 0.95, 1.0]))
offset_rows = calibration[calibration.mode.eq('draw_offset')]
best_offsets = offset_rows.loc[offset_rows.groupby('source_rollout_id').first10_active_rms.idxmin()]
display(best_offsets[['suite', 'task_idx', 'episode_idx', 'chunk_idx',
                      'draw_offset', 'first10_active_rms']])
print('best-offset counts:', best_offsets.draw_offset.value_counts().sort_index().to_dict())